#Data Analysis Project to Analyse New York City Restaurant Inspection Results

This notebook documents the data cleaning, analysis, and visualization process using pandas and Streamlit.

In [ ]:
import pandas as pd

df = pd.read_csv('NYResturantRaw.csv') #import raw csv file

'''----- FUNCTION TO CLEAN DATA -----'''
def clean_data(df):
    df['CAMIS'] = pd.to_numeric(df['CAMIS'], errors='coerce') 
    df['ZIPCODE'] = pd.to_numeric(df['ZIPCODE'], errors='coerce')
    df['PHONE'] = pd.to_numeric(df['PHONE'], errors='coerce')
    df['INSPECTION_DATE'] = pd.to_datetime(df['INSPECTION_DATE'], errors='coerce')
    df['BUILDING'] = pd.to_numeric(df['BUILDING'], errors='coerce')
    df = df[df['INSPECTION_DATE'] != pd.Timestamp('1900-01-01')]
    df.to_csv('NYResturantClean.csv', index = False)

    return df

clean_data(df)


#Data Cleansing

This first section is to clean the data.

The data needs cleaning because values are stored as text and needs to be converted to dates and numbers so that it can be processed correctly.

Any inspections with a date of 01/01/1900 will be dropped because that means no inspection has taken place.

The cleansed data is then saved in a new csv file so that the original data can still be preserved.


#Data Processing

Once cleansed the data can be processed and a dashboard can be created:

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px

df = pd.read_csv("NYResturantClean.csv")

st.set_page_config(page_title="NYC Restaurant Dashboard", layout="wide")

This configures streamlit, pandas and plotly in order to process the data.

Then the cleaned data is loaded into the file to be processed.

Then the page is configured to create a StreamLit dashboard.

In [ ]:
# --- Sidebar Filters ---
st.sidebar.header("Filter the data")

# Create a list of boroughs with "All" at the top
boroughs = ['All'] + sorted(df['BORO'].dropna().unique().tolist())
borough = st.sidebar.selectbox("Select Borough", boroughs)

# Multiselect for grades
grade = st.sidebar.multiselect(
    "Select Grade", 
    sorted(df['GRADE'].dropna().unique()), 
    default=sorted(df['GRADE'].dropna().unique())
)

# --- Apply Filters ---
# Borough filter
if borough == 'All':
    filtered_df = df.copy()
else:
    filtered_df = df[df['BORO'] == borough]

# Grade filter (applied after borough)
filtered_df = filtered_df[filtered_df['GRADE'].isin(grade)]


This is the sidebar being intialised to filter by borough and grade given to each resturant.

In [ ]:
# --- Dashboard Header ---
st.title("NYC Restaurant Inspection Dashboard")
st.markdown("Explore health inspection data for restaurants across New York City.")

# --- Metric Summary ---
st.metric("Restaurants Shown", value=len(filtered_df))

This initialises the dashboard header and the metric summary so the number of resturants shown can be seen

In [ ]:
# --- Cuisine Distribution Chart ---
st.subheader("Top 10 Cuisines")
top_cuisines = filtered_df['CUISINE_DESCRIPTION'].value_counts().head(10)
fig1 = px.bar(top_cuisines, title="Top 10 Cuisines in Selected Borough", labels={'index':'Cuisine', 'value':'Count'})
st.plotly_chart(fig1)

This graph shows the top 10 most popular cuisines in the area, with American being one of the most popular across the region.

In [ ]:
# --- Grade Pie Chart ---
st.subheader("Grade Distribution")
fig2 = px.pie(filtered_df, names='GRADE', title='Distribution of Grades')
st.plotly_chart(fig2)

This is a pie chart that shows distribution of grades given to each resturant as with the highest grade being A.

In [ ]:
# --- Grade Distribution Chart ---
st.subheader("Top 10 Cuisines")
top_cuisines = filtered_df['GRADE'].value_counts().head(10)
fig4 = px.bar(top_cuisines, title="Grade Distribution", labels={'index':'Grade', 'value':'Count'})
st.plotly_chart(fig4)

This creates a bar chart that shows the distributions of the grades as well to see how well the resturants in that area does

In [ ]:
critical_df = df[df['CRITICAL_FLAG'] == 'Critical']

fig = px.scatter_mapbox(
    critical_df,
    lat="Latitude",
    lon="Longitude",
    hover_name="DBA",  # restaurant name
    hover_data=["BORO"],
    zoom=9.5,
    height=600
)

fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

st.subheader("Interactive Map of Critical Violations")
st.plotly_chart(fig)

This creates a map that shows the longitude and latitude of resturants that have recieved a critical flag which means there was a critical issue with that particular resturant.

In [ ]:
# --- Data Table ---
st.subheader("Filtered Data Table")
st.dataframe(filtered_df)

This shows the data in table format as well.